# Predicción de diabetes tipo 2: revisión y despliegue del modelo
## Proyecto de aula de Minería de Datos, tercer corte (Actividad 1)

**Universidad de la Costa (CUC), Departamento de Ciencias de la Computación y Electrónica**

**Integrantes:** Esteban Lopez; Gonzalez Ramirez Harold

**Tipo de proyecto:** clasificación binaria. **Variable objetivo:** `Outcome` (1 = tiene diabetes, 0 = no).

**Pregunta:** ¿se puede predecir si una paciente tiene diabetes a partir de ocho variables clínicas y demográficas, cuidando sobre todo de no dejar pasar a las pacientes que sí la tienen?

**Datos:** Pima Indians Diabetes Database (Smith et al., 1988), 768 pacientes y 8 predictoras. Fuente original: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database. El archivo que usamos es una copia sin encabezados descargada de https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv, a la que agregamos los nombres de columna siguiendo el orden documentado del dataset (no la descargamos directamente de Kaggle).

**Qué hace este notebook.** Siguiendo la actividad, no repite el EDA. (1) Reconstruye el pipeline y los resultados del modelo con la misma semilla y partición. (2) Los revisa con la ficha de calidad antes de llevarlos a Streamlit. (3) Exporta el pipeline completo y las métricas que usa la aplicación.

## 1. Datos y preparación (resumen de lo ya trabajado)

In [1]:
import os
import json
import joblib
import sklearn
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)

SEMILLA = 42   # misma semilla que en el segundo corte

In [2]:
URL = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
COLUMNAS = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin",
            "BMI", "DiabetesPedigreeFunction", "Age", "Outcome"]

# Si el archivo está en el repositorio lo usamos; si no, lo descargamos (la copia de la URL no trae encabezados)
if os.path.exists("data/diabetes.csv"):
    df = pd.read_csv("data/diabetes.csv")
elif os.path.exists("diabetes.csv"):
    df = pd.read_csv("diabetes.csv")
else:
    df = pd.read_csv(URL, header=None, names=COLUMNAS)

assert df.shape == (768, 9), "El dataset no tiene la forma esperada"
print("Filas y columnas:", df.shape, "| nulos:", df.isnull().sum().sum(), "| duplicados:", df.duplicated().sum())

# Ceros que no pueden ser reales (mediciones no realizadas) -> NaN. No usa información de otras filas.
cols_con_ceros = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("\nCeros imposibles por variable:")
print((df[cols_con_ceros] == 0).sum().to_string())

X = df.drop(columns="Outcome").copy()
y = df["Outcome"]
X[cols_con_ceros] = X[cols_con_ceros].replace(0, np.nan)

# 80 % entrenamiento / 20 % prueba, estratificado
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEMILLA)
print("\nEntrenamiento:", X_train.shape, "| Prueba:", X_test.shape)

Filas y columnas: (768, 9) | nulos: 0 | duplicados: 0

Ceros imposibles por variable:
Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11

Entrenamiento: (614, 8) | Prueba: (154, 8)


Los 374 ceros de `Insulin`, 227 de `SkinThickness`, 35 de `BloodPressure`, 11 de `BMI` y 5 de `Glucose` se tratan como datos no medidos. La imputación con la mediana no se hace aquí sino dentro del `Pipeline`, para que se aprenda solo con los datos de entrenamiento y no haya fuga de información.

## 2. Pipeline y resultados del modelo (reconstrucción del segundo corte)

In [3]:
def crear_pipeline(modelo, escalar):
    # La imputación (y el escalado) se aprenden SOLO con los datos de entrenamiento
    pasos = [("imputar", SimpleImputer(strategy="median"))]
    if escalar:
        pasos.append(("escalar", StandardScaler()))
    pasos.append(("modelo", modelo))
    return Pipeline(pasos)

modelos = {
    "Regresión logística": crear_pipeline(LogisticRegression(max_iter=1000, random_state=SEMILLA), True),
    "Árbol de decisión (profundidad 4)": crear_pipeline(DecisionTreeClassifier(max_depth=4, random_state=SEMILLA), False),
    "Regresión logística balanceada": crear_pipeline(LogisticRegression(max_iter=1000, random_state=SEMILLA, class_weight="balanced"), True),
}

predicciones, probabilidades, filas = {}, {}, []
for nombre, p in modelos.items():
    p.fit(X_train, y_train)
    predicciones[nombre] = p.predict(X_test)
    probabilidades[nombre] = p.predict_proba(X_test)[:, 1]
    pred = predicciones[nombre]
    filas.append({"Modelo": nombre,
                  "Exactitud entrenamiento": accuracy_score(y_train, p.predict(X_train)),
                  "Exactitud prueba": accuracy_score(y_test, pred),
                  "Precisión (clase 1)": precision_score(y_test, pred),
                  "Recall (clase 1)": recall_score(y_test, pred),
                  "F1 (clase 1)": f1_score(y_test, pred),
                  "AUC": roc_auc_score(y_test, probabilidades[nombre])})
tabla_test = pd.DataFrame(filas).set_index("Modelo").round(3)
tabla_test

,Exactitud entrenamiento,Exactitud prueba,Precisión (clase 1),Recall (clase 1),F1 (clase 1),AUC
Modelo,,,,,,
Regresión logística,0.796,0.708,0.600,0.500,0.545,0.813
Árbol de decisión (profundidad 4),0.795,0.786,0.698,0.685,0.692,0.789
Regresión logística balanceada,0.759,0.734,0.603,0.704,0.650,0.813


In [4]:
# Validación cruzada estratificada de 5 pliegues sobre todos los datos (imputador y escalador se reajustan en cada pliegue)
validacion = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
nombres_metricas = ["accuracy", "precision", "recall", "f1", "roc_auc"]
resultados_cv = {n: cross_validate(p, X, y, cv=validacion, scoring=nombres_metricas) for n, p in modelos.items()}

media = pd.DataFrame({n: {m: r["test_" + m].mean() for m in nombres_metricas} for n, r in resultados_cv.items()}).T
desv = pd.DataFrame({n: {m: r["test_" + m].std() for m in nombres_metricas} for n, r in resultados_cv.items()}).T
formato = "{:.3f}".format
tabla_cv = media.apply(lambda col: col.map(formato)) + " ± " + desv.apply(lambda col: col.map(formato))
tabla_cv.columns = ["Exactitud", "Precisión", "Recall", "F1", "AUC"]
tabla_cv

,Exactitud,Precisión,Recall,F1,AUC
Regresión logística,0.772 ± 0.017,0.726 ± 0.062,0.571 ± 0.047,0.636 ± 0.021,0.837 ± 0.020
Árbol de decisión (profundidad 4),0.728 ± 0.028,0.612 ± 0.047,0.638 ± 0.109,0.617 ± 0.045,0.768 ± 0.027
Regresión logística balanceada,0.747 ± 0.040,0.623 ± 0.063,0.727 ± 0.047,0.669 ± 0.040,0.836 ± 0.020


**Por qué la regresión logística balanceada.** Con una sola partición el árbol de profundidad 4 parece mejor (exactitud 0.786 frente a 0.708 de la logística), pero esa partición es solo una de muchas posibles. En validación cruzada la logística tiene mejor AUC (0.837 frente a 0.768) y el árbol tiene un recall muy inestable (0.638 ± 0.109). La versión balanceada logra el mejor recall (0.727 ± 0.047) y el mejor F1 (0.669), con el mismo AUC que la logística normal (0.836): no discrimina mejor, pero mueve el punto de corte para detectar más pacientes con diabetes, que es lo que más importa en un tamizaje. Pagamos con más falsos positivos y menos exactitud (0.747 frente a 0.772).

## 3. Revisión del modelo antes del despliegue (ficha de calidad)

In [5]:
NOMBRE_FINAL = "Regresión logística balanceada"
modelo_final = modelos[NOMBRE_FINAL]
p_final = predicciones[NOMBRE_FINAL]

tn, fp, fn, tp = confusion_matrix(y_test, p_final).ravel()
print(f"Matriz de confusión: VN={tn}, FP={fp}, FN={fn}, VP={tp}")
print(classification_report(y_test, p_final, target_names=["Sin diabetes (0)", "Con diabetes (1)"], digits=3))

# Alerta de la actividad: exactitud o métrica principal por debajo de 0.70
UMBRAL = 0.70
acc_test = accuracy_score(y_test, p_final)
acc_cv = resultados_cv[NOMBRE_FINAL]["test_accuracy"].mean()
rec_pliegues = resultados_cv[NOMBRE_FINAL]["test_recall"]
print(f"Exactitud en prueba: {acc_test:.3f} -> {'supera' if acc_test >= UMBRAL else 'REVISAR, está por debajo de'} {UMBRAL}")
print(f"Exactitud media en validación cruzada: {acc_cv:.3f} -> {'supera' if acc_cv >= UMBRAL else 'REVISAR, está por debajo de'} {UMBRAL}")
print(f"Recall (clase 1) en prueba: {recall_score(y_test, p_final):.3f} | falsos negativos: {fn} de {fn + tp}")
print("Recall en cada pliegue:", np.round(rec_pliegues, 3), "| pliegues bajo 0.70:", int((rec_pliegues < UMBRAL).sum()), "de", len(rec_pliegues))

Matriz de confusión: VN=75, FP=25, FN=16, VP=38
                  precision    recall  f1-score   support

Sin diabetes (0)      0.824     0.750     0.785       100
Con diabetes (1)      0.603     0.704     0.650        54

        accuracy                          0.734       154
       macro avg      0.714     0.727     0.717       154
    weighted avg      0.747     0.734     0.738       154

Exactitud en prueba: 0.734 -> supera 0.7
Exactitud media en validación cruzada: 0.747 -> supera 0.7
Recall (clase 1) en prueba: 0.704 | falsos negativos: 16 de 54
Recall en cada pliegue: [0.741 0.722 0.778 0.642 0.755] | pliegues bajo 0.70: 1 de 5


La exactitud en prueba (0.734) y en validación cruzada (0.747) superan 0.70, así que la alerta por exactitud no se activa. Pero para este problema la métrica principal es el recall de la clase "con diabetes", y ahí el margen es corto: 0.704 en prueba y, por pliegue, 0.741, 0.722, 0.778, 0.642, 0.755. **1 de los 5 pliegues queda por debajo de 0.70**, y el modelo deja pasar 16 de las 54 pacientes con diabetes del conjunto de prueba. Por eso hicimos la revisión completa y no nos quedamos con que "pasa el 70 %".

In [6]:
# 1) Separación: ninguna paciente puede estar en entrenamiento y prueba a la vez
compartidas = set(X_train.index) & set(X_test.index)
assert len(compartidas) == 0
print("Pacientes en ambos conjuntos:", len(compartidas))

# 2) Fuga de información: imputador y escalador son pasos del pipeline, que se ajustó solo con X_train
print("Pasos del pipeline final:", list(modelo_final.named_steps))

# 3) Desbalance
print(f"% con diabetes -> entrenamiento: {y_train.mean() * 100:.1f} | prueba: {y_test.mean() * 100:.1f}")

Pacientes en ambos conjuntos: 0
Pasos del pipeline final: ['imputar', 'escalar', 'modelo']
% con diabetes -> entrenamiento: 34.9 | prueba: 35.1


In [7]:
# 4) Colinealidad: factor de inflación de la varianza (VIF), con los datos de entrenamiento imputados
X_imp = pd.DataFrame(modelo_final.named_steps["imputar"].transform(X_train), columns=X.columns)
vif = {}
for col in X_imp.columns:
    otras = X_imp.drop(columns=col)
    r2 = LinearRegression().fit(otras, X_imp[col]).score(otras, X_imp[col])
    vif[col] = 1 / (1 - r2)
print(pd.Series(vif).round(2).sort_values(ascending=False).to_string())

Age                         1.60
BMI                         1.55
SkinThickness               1.46
Pregnancies                 1.42
Glucose                     1.40
Insulin                     1.31
BloodPressure               1.22
DiabetesPedigreeFunction    1.07


In [8]:
# 5) Sobreajuste: entrenamiento vs prueba del modelo final
pd.DataFrame({
    "entrenamiento": [accuracy_score(y_train, modelo_final.predict(X_train)), recall_score(y_train, modelo_final.predict(X_train))],
    "prueba": [accuracy_score(y_test, p_final), recall_score(y_test, p_final)],
}, index=["exactitud", "recall (clase 1)"]).round(3)

,entrenamiento,prueba
exactitud,0.759,0.734
recall (clase 1),0.710,0.704


In [9]:
# 6) Hiperparámetros: ¿cambia mucho el resultado si movemos la regularización C?
# Solo validación cruzada sobre el ENTRENAMIENTO; el conjunto de prueba no se toca.
busqueda = GridSearchCV(clone(modelo_final), {"modelo__C": [0.01, 0.1, 1, 10, 100]}, cv=validacion,
                        scoring=["recall", "f1", "roc_auc"], refit="f1")
busqueda.fit(X_train, y_train)
tabla_C = pd.DataFrame(busqueda.cv_results_)[["param_modelo__C", "mean_test_recall", "mean_test_f1", "mean_test_roc_auc"]]
tabla_C.columns = ["C", "recall", "F1", "AUC"]
tabla_C.round(3)

,C,recall,F1,AUC
0,0.01,0.710,0.667,0.841
1,0.10,0.720,0.682,0.845
2,1.00,0.701,0.672,0.844
3,10.00,0.701,0.672,0.844
4,100.00,0.701,0.672,0.844


| Punto de la ficha | Qué revisamos | Resultado |
|---|---|---|
| Tratamiento de los datos | Ceros imposibles a `NaN`; imputación con la mediana dentro del pipeline | No se eliminaron filas |
| Separación entrenamiento/prueba | 80/20 estratificada, semilla 42 | 0 pacientes en ambos conjuntos; 34.9 % y 35.1 % de positivos |
| Validación cruzada | 5 pliegues estratificados | Recall 0.727 ± 0.047; exactitud 0.747 ± 0.040 |
| Fuga de información | Imputador y escalador dentro del `Pipeline` | Se ajustan solo con entrenamiento (y con cada pliegue en la validación cruzada) |
| Desbalance | 34.9 % de positivos | `class_weight="balanced"` y métricas por clase |
| Colinealidad | VIF de las 8 predictoras | Máximo 1.60 (`Age`), muy por debajo de 5 |
| Sobreajuste | Entrenamiento frente a prueba | Exactitud 0.759 frente a 0.734; recall 0.710 frente a 0.704; sin sobreajuste |
| Hiperparámetros | Búsqueda de `C` solo con el entrenamiento | Ver tabla y decisión abajo |

**Decisión sobre el hiperparámetro.** La búsqueda de `C` da recall entre 0.701 y 0.720, F1 entre 0.667 y 0.682 y AUC entre 0.841 y 0.845. El mejor F1 está en `C = 0.1` (0.682) y el de `C = 1`, el valor por defecto que ya usábamos, es 0.672. Esa diferencia de 0.010 es mucho menor que la desviación entre pliegues (cercana a 0.04), así que no podemos distinguirla del azar.

**No cambiamos el modelo.** Seguimos con `C = 1` para no ajustar un hiperparámetro por una diferencia dentro del ruido. Como no hubo correcciones ni optimización, las métricas de antes y después son las mismas (sección 2).

## 4. Exportación del pipeline para la aplicación

In [10]:
os.makedirs("models", exist_ok=True)
os.makedirs("data", exist_ok=True)

# 1) Pipeline completo (imputación + escalado + modelo)
joblib.dump(modelo_final, "models/pipeline_diabetes.joblib")

# 2) Registros de prueba con la predicción de este notebook (para comprobar que la app da lo mismo)
prueba = X_test.copy()
prueba.insert(0, "id_original", X_test.index)
prueba["Outcome_real"] = y_test.values
prueba["prediccion_notebook"] = modelo_final.predict(X_test)
prueba["probabilidad_notebook"] = modelo_final.predict_proba(X_test)[:, 1]
prueba.to_csv("data/conjunto_prueba.csv", index=False)

# 3) Métricas finales que muestra la app (salen de las mismas tablas de arriba)
informe = classification_report(y_test, p_final, output_dict=True)
metricas_finales = {
    "modelo": NOMBRE_FINAL,
    "version_sklearn": sklearn.__version__,
    "n_entrenamiento": int(len(X_train)),
    "n_prueba": int(len(X_test)),
    "positivos_prueba": int(y_test.sum()),
    "prueba": {
        "exactitud": float(accuracy_score(y_test, p_final)),
        "precision_clase1": float(precision_score(y_test, p_final)),
        "recall_clase1": float(recall_score(y_test, p_final)),
        "f1_clase1": float(f1_score(y_test, p_final)),
        "f1_macro": float(f1_score(y_test, p_final, average="macro")),
        "auc": float(roc_auc_score(y_test, probabilidades[NOMBRE_FINAL])),
    },
    "matriz_confusion": confusion_matrix(y_test, p_final).tolist(),   # [[VN, FP], [FN, VP]]
    "por_clase": {k: {m: float(v) for m, v in informe[k].items()} for k in ["0", "1"]},
    "validacion_cruzada": {m: {"media": float(media.loc[NOMBRE_FINAL, m]), "desviacion": float(desv.loc[NOMBRE_FINAL, m])} for m in nombres_metricas},
    "recall_por_pliegue": [float(v) for v in rec_pliegues],
    "comparacion_modelos_cv": {n: {m: float(media.loc[n, m]) for m in nombres_metricas} for n in media.index},
}
with open("models/metricas_finales.json", "w", encoding="utf-8") as archivo:
    json.dump(metricas_finales, archivo, indent=2, ensure_ascii=False)

print("Archivos generados:", sorted(os.listdir("models")), sorted(os.listdir("data")))

Archivos generados: ['metricas_finales.json', 'pipeline_diabetes.joblib'] ['conjunto_prueba.csv', 'diabetes.csv']


In [11]:
# Comprobación: el pipeline guardado, al cargarse de nuevo, predice exactamente lo mismo que el original
cargado = joblib.load("models/pipeline_diabetes.joblib")
assert np.array_equal(cargado.predict(X_test), modelo_final.predict(X_test))
assert np.allclose(cargado.predict_proba(X_test), modelo_final.predict_proba(X_test))
print("OK: el pipeline guardado da las mismas predicciones que el original sobre las", len(X_test), "pacientes de prueba.")

print("\nRegistro de prueba 0 (fila 0 de data/conjunto_prueba.csv), el que se usa para comparar con la app:")
print(prueba.iloc[0].to_string())

OK: el pipeline guardado da las mismas predicciones que el original sobre las 154 pacientes de prueba.

Registro de prueba 0 (fila 0 de data/conjunto_prueba.csv), el que se usa para comparar con la app:
id_original                  44.0000
Pregnancies                   7.0000
Glucose                     159.0000
BloodPressure                64.0000
SkinThickness                    NaN
Insulin                          NaN
BMI                          27.4000
DiabetesPedigreeFunction      0.2940
Age                          40.0000
Outcome_real                  0.0000
prediccion_notebook           1.0000
probabilidad_notebook         0.7332


## 5. Uso de inteligencia artificial

La evaluación del uso de IA para esta actividad (herramienta, propósito, prompt, verificación, problemas y ajustes) está en el `README.md` del repositorio, como pide la actividad.